# 05 - Kuantifikasi Pergeseran Distribusi (Section 3.8 Paper 4)

Mengukur seberapa besar pergeseran distribusi CIC-IDS2017 vs CSE-CIC-IDS2018 pada 76 fitur
irisan, untuk mendukung klaim diagnosis (bukan sekadar bukti tidak-langsung).

Metrik: (1) jarak 1-Wasserstein (W1) per fitur di ruang z-score bersama + rerata;
(2) akurasi domain-classifier (bisa/tidak membedakan asal dataset) + Proxy A-distance.

Dijalankan di AWS SageMaker. Baca artefak `cic2017.pkl`/`cic2018.pkl` (hasil notebook 01),
tulis `shift_results.json`, upload ke S3. Semua angka dari data nyata.

In [ ]:
import sys
!{sys.executable} -m pip install -q numpy pandas scipy scikit-learn
print('deps OK')

In [ ]:
import os, json, pickle, subprocess
import numpy as np
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

ART='./artifacts'
S3_ARTIFACTS_OUT='s3://ssh-detection-features-232032302717/paper4-xai/artifacts/'
SEED=42; np.random.seed(SEED)
os.makedirs(ART, exist_ok=True)
# selalu tarik artefak terbaru (butuh cic2017.pkl & cic2018.pkl dari notebook 01)
subprocess.run(['aws','s3','sync',S3_ARTIFACTS_OUT,ART,'--only-show-errors'], check=True)
ref=pickle.load(open(f'{ART}/cic2017.pkl','rb'))   # CIC-IDS2017
tgt=pickle.load(open(f'{ART}/cic2018.pkl','rb'))   # CSE-CIC-IDS2018
FEATURES=ref['features']
Xr, Xt = ref['X'].values.astype(float), tgt['X'].values.astype(float)
print('CIC2017:', Xr.shape, '| CIC2018:', Xt.shape, '| fitur:', len(FEATURES))

In [ ]:
# === W1 per fitur di ruang z-score BERSAMA (fit scaler pd gabungan) ===
# Batasi sampel utk kecepatan (W1 1D stabil pd puluhan ribu sampel).
NMAX=50000
def sub(X):
    if len(X)>NMAX:
        idx=np.random.RandomState(SEED).choice(len(X),NMAX,replace=False); return X[idx]
    return X
Xr_s, Xt_s = sub(Xr), sub(Xt)
scaler=StandardScaler().fit(np.vstack([Xr_s,Xt_s]))
Zr, Zt = scaler.transform(Xr_s), scaler.transform(Xt_s)
w1=[]
for j in range(len(FEATURES)):
    a=Zr[:,j]; b=Zt[:,j]
    a=a[np.isfinite(a)]; b=b[np.isfinite(b)]
    if len(a)==0 or len(b)==0: w1.append(0.0); continue
    w1.append(float(wasserstein_distance(a,b)))
w1=np.array(w1)
order=np.argsort(w1)[::-1]
top=[(FEATURES[i], round(float(w1[i]),4)) for i in order[:10]]
print('Rerata W1 (z-space):', round(float(w1.mean()),4))
print('Top-10 fitur bergeser:')
for f,v in top: print(f'  {f:32} {v}')

In [ ]:
# === Domain-classifier: bisakah membedakan 2017 vs 2018? ===
# label domain: 0=CIC2017, 1=CIC2018. Fitur = z-space bersama. 5-fold CV.
Xdom=np.vstack([Zr,Zt]); ydom=np.r_[np.zeros(len(Zr)),np.ones(len(Zt))]
clf=RandomForestClassifier(n_estimators=100,max_depth=12,n_jobs=-1,random_state=SEED)
acc=cross_val_score(clf,Xdom,ydom,cv=5,scoring='accuracy',n_jobs=-1)
dom_acc=float(acc.mean())
# Proxy A-distance: d_A = 2(1-2*error) = 2(2*acc-1)
err=1-dom_acc; proxy_A=float(2*(1-2*err))
print('Akurasi domain-classifier (5-fold):', round(dom_acc,4), '+/-', round(float(acc.std()),4))
print('Proxy A-distance:', round(proxy_A,4))
print('Interpretasi: acc ~1.0 & d_A ~2.0 => dua domain sangat mudah dibedakan = shift besar.')

In [ ]:
# === Simpan artefak + upload S3 ===
out={
  'ref_dataset':'CIC-IDS2017','tgt_dataset':'CSE-CIC-IDS2018',
  'n_features':len(FEATURES),'n_sample_per_domain':int(min(len(Xr),NMAX)),
  'w1_mean':round(float(w1.mean()),4),
  'w1_median':round(float(np.median(w1)),4),
  'w1_max':round(float(w1.max()),4),
  'w1_top10':[{'feature':FEATURES[i],'w1':round(float(w1[i]),4)} for i in order[:10]],
  'w1_per_feature':{FEATURES[i]:round(float(w1[i]),4) for i in range(len(FEATURES))},
  'domain_clf_acc':round(dom_acc,4),
  'domain_clf_acc_std':round(float(acc.std()),4),
  'proxy_a_distance':round(proxy_A,4),
  'seed':SEED
}
json.dump(out, open(f'{ART}/shift_results.json','w'), indent=2)
subprocess.run(['aws','s3','sync',ART,S3_ARTIFACTS_OUT,'--only-show-errors','--exclude','*','--include','shift_results.json'], check=True)
print('shift_results.json tersimpan & ter-upload')
print(json.dumps({k:v for k,v in out.items() if k not in ('w1_per_feature',)}, indent=2))